# 🛠️ Practice Project Template 06: End-to-End RAG Document Assistant
## Hands-On Implementation: Retrieval-Augmented Generation & Vector Search

In this project, you will build a complete, runnable **Retrieval-Augmented Generation (RAG) System** from scratch.
You will see how modern AI assistants ingest documents, convert them into vector embeddings, retrieve the most relevant passages for a user query, and synthesize a grounded prompt that prevents hallucinations.

---

### 📋 Project Roadmap:
1. **Document Ingestion**: Create an internal corporate/research knowledge base.
2. **Chunking**: Implement sliding-window text chunking with configurable overlap.
3. **Vector Embeddings**: Implement dense vector representation and Cosine Similarity.
4. **In-Memory Vector Store**: Build an indexed vector store supporting top-$k$ nearest-neighbor retrieval.
5. **Prompt Augmentation**: Format a context-injected prompt template for LLMs.
6. **Hallucination Verification**: Test the system with questions inside and outside the knowledge base!


In [ ]:
import numpy as np
import re
from typing import List, Dict, Any

print("RAG Project Template initialized!")


---
## Step 1: Knowledge Base Documents
Here is our corpus of internal research guidelines:


In [ ]:
documents = [
    {
        "id": "KB_001",
        "title": "Learning Rate Schedules",
        "text": "For training modern Transformers, cosine annealing with a linear warmup over the first 5-10% of training steps is standard. The peak learning rate for AdamW typically ranges from 1e-4 to 5e-5."
    },
    {
        "id": "KB_002",
        "title": "Gradient Clipping Strategies",
        "text": "To prevent exploding gradients in recurrent networks and deep transformers, gradient norm clipping is applied before the optimizer step. A max norm of 1.0 or 5.0 is recommended."
    },
    {
        "id": "KB_003",
        "title": "Precision-Recall AUC Guidelines",
        "text": "When class imbalance exceeds 90:10, ROC-AUC gives an overly optimistic assessment due to the large number of true negatives. PR-AUC and Macro F1 must be used instead."
    }
]

print(f"Loaded {len(documents)} knowledge base documents.")


---
## Step 2: [YOUR TURN] Implement Sliding-Window Chunking
**Instructions:**
- Complete the `chunk_text(text, max_words=25, overlap=5)` function.
- It should split words and return a list of overlapping text chunks.


In [ ]:
def chunk_text(text: str, max_words: int = 25, overlap: int = 5) -> List[str]:
    ### YOUR CODE HERE ###
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += max_words - overlap
        if end >= len(words):
            break
    return chunks

# Test chunking
all_chunks = []
for doc in documents:
    c_list = chunk_text(doc["text"], max_words=20, overlap=5)
    for idx, c in enumerate(c_list):
        all_chunks.append({
            "chunk_id": f"{doc['id']}_c{idx}",
            "doc_title": doc["title"],
            "text": c
        })

print(f"Generated {len(all_chunks)} chunks.")
print("Sample chunk:", all_chunks[0])


---
## Step 3: [YOUR TURN] Build Vector Embedder & Cosine Similarity
**Instructions:**
- Implement `cosine_similarity(vec_a, vec_b)`.
- Use the subword hashing embedder to convert text into a 64-dimensional dense vector.


In [ ]:
def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    ### YOUR CODE HERE ###
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return float(np.dot(u, v) / (norm_u * norm_v))

class DenseEmbedder:
    def __init__(self, dim: int = 64):
        self.dim = dim
        
    def embed(self, text: str) -> np.ndarray:
        words = re.findall(r'\w+', text.lower())
        vec = np.zeros(self.dim, dtype=np.float32)
        for w in words:
            vec[hash(w) % self.dim] += 1.0
            for i in range(len(w) - 2):
                vec[hash(w[i:i+3]) % self.dim] += 0.5
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec /= norm
        return vec

embedder = DenseEmbedder(dim=64)
test_vec = embedder.embed("Transformers use cosine annealing")
print(f"Embedding shape: {test_vec.shape}, L2 Norm: {np.linalg.norm(test_vec):.4f}")


---
## Step 4: [YOUR TURN] In-Memory Vector Store & Top-K Search
**Instructions:**
- Build `VectorStore` class with `add()` and `search(query_vec, top_k=2)` methods.


In [ ]:
class VectorStore:
    def __init__(self):
        self.entries = []
        
    def add(self, chunk_id: str, title: str, text: str, vector: np.ndarray):
        self.entries.append({
            "chunk_id": chunk_id,
            "title": title,
            "text": text,
            "vector": vector
        })
        
    def search(self, query_vector: np.ndarray, top_k: int = 2) -> List[Dict[str, Any]]:
        ### YOUR CODE HERE: Compute cosine similarity for each entry and return top_k ###
        scored = []
        for e in self.entries:
            sim = cosine_similarity(query_vector, e["vector"])
            scored.append((sim, e))
            
        scored.sort(key=lambda x: x[0], reverse=True)
        return [{"score": s, **item} for s, item in scored[:top_k]]

db = VectorStore()
for c in all_chunks:
    v = embedder.embed(c["text"])
    db.add(c["chunk_id"], c["doc_title"], c["text"], v)

print(f"VectorStore ready with {len(db.entries)} items.")


---
## Step 5: RAG Prompt Synthesis & Query Pipeline
Let's assemble the retrieval step and prompt synthesizer!


In [ ]:
def answer_question(user_question: str, top_k: int = 1):
    q_vec = embedder.embed(user_question)
    retrieved_items = db.search(q_vec, top_k=top_k)
    
    context_text = "\n".join([f"- [{r['title']}]: {r['text']}" for r in retrieved_items])
    
    prompt = f"""System: Answer the question using strictly the verified context below. If not found, output 'Unknown'.

Verified Context:
{context_text}

User Question: {user_question}
Answer:"""
    
    print("=== RETRIEVAL RESULTS ===")
    for r in retrieved_items:
        print(f"Top Match: [{r['title']}] (Similarity: {r['score']:.4f})")
    print("\n=== GENERATED GROUNDED PROMPT ===")
    print(prompt)
    print("-" * 50)

# Test Grounded Queries
answer_question("What learning rate schedule is recommended for transformers?", top_k=1)
answer_question("When should we avoid ROC-AUC?", top_k=1)


---
## 🏆 Practice Challenge Extensions:
1. **Threshold Filtering**: Add a minimum similarity threshold (e.g. `min_score = 0.40`). If all retrieved chunks score below it, automatically declare "No relevant context found".
2. **Hybrid Search**: Combine keyword matching (BM25) with dense vector search to get the benefits of both!
